In [1]:
import sys
!{sys.executable} -m pip install yfinance

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
sys.path.insert(0, "..")

from src.data.price_loader import download_prices, get_log_returns

prices = download_prices("AAPL", "2023-01-01", "2023-03-31")
print(prices.columns.tolist())
print(prices.shape)

returns = get_log_returns(prices)
print(returns.head())

['Close', 'High', 'Low', 'Open', 'Volume']
(61, 5)
Date
2023-01-04    0.010261
2023-01-05   -0.010661
2023-01-06    0.036133
2023-01-09    0.004080
2023-01-10    0.004447
Name: Close, dtype: float64


In [3]:
import pandas as pd
from src.data.event_builder import (
    UNIVERSE, ANALYSIS_START, ANALYSIS_END,
    get_pre_window, get_post_window, compute_abnormal_return
)
from src.data.price_loader import download_prices, get_log_returns

# Quick sanity check on windowing
prices = download_prices("AAPL", "2022-01-01", "2023-06-30")
returns = get_log_returns(prices)

# Simulate an event on 2023-02-02 (AAPL Q4 2022 earnings)
event_date = pd.Timestamp("2023-02-02")
pre  = get_pre_window(returns, event_date, window=30)
post = get_post_window(returns, event_date, window=5)

print("Pre window:", pre.index[0].date(), "→", pre.index[-1].date(), f"({len(pre)} days)")
print("Post window:", post.index[0].date(), "→", post.index[-1].date(), f"({len(post)} days)")
print("Overlap check (must be 0):", len(set(pre.index) & set(post.index)))

Pre window: 2022-12-19 → 2023-02-01 (30 days)
Post window: 2023-02-02 → 2023-02-08 (5 days)
Overlap check (must be 0): 0


In [4]:
import sys
import numpy as np
sys.path.insert(0, "..")

from src.fourier.spectral import compute_spectral_features

# Use the pre-window we already computed
features = compute_spectral_features(pre.values)

print(f"Spectral entropy:   {features['spectral_entropy']:.4f}")
print(f"Dominant frequency: {features['dominant_frequency']:.4f}")
print(f"Dominant power:     {features['dominant_power']:.4f}")
print(f"N observations:     {features['n_obs']}")

Spectral entropy:   2.4819
Dominant frequency: 0.3667
Dominant power:     0.2130
N observations:     30


In [5]:
from src.fourier.features import extract_features_for_event
from src.data.price_loader import download_prices, get_log_returns

spy_prices  = download_prices("SPY", "2022-01-01", "2023-06-30")
spy_returns = get_log_returns(spy_prices)

result = extract_features_for_event(
    ticker="AAPL",
    event_date=pd.Timestamp("2023-02-02"),
    stock_returns=returns,
    benchmark_returns=spy_returns,
)

print(result)

{'ticker': 'AAPL', 'event_date': Timestamp('2023-02-02 00:00:00'), 'spectral_entropy': np.float64(2.481869841110682), 'dominant_frequency': np.float64(0.36666666666666664), 'dominant_power': np.float64(0.2130066982729888), 'cumulative_abnormal_return': np.float64(0.04402431056313236), 'day1_abnormal_return': np.float64(0.021940221853018388), 'pre_window_size': 30, 'post_window_size': 5}


In [6]:
import sys
!{sys.executable} -m pip install lxml

import yfinance as yf
ticker = yf.Ticker("AAPL")
dates = ticker.earnings_dates
print(dates)
print(dates.shape)

Defaulting to user installation because normal site-packages is not writeable
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-04-30 16:00:00-04:00          1.94           NaN          NaN
2026-01-29 16:00:00-05:00          2.67          2.84         6.25
2025-10-30 16:00:00-04:00          1.77          1.85         4.52
2025-07-31 16:00:00-04:00          1.43          1.57         9.48
2025-05-01 16:00:00-04:00          1.63          1.65         1.50
2025-01-30 16:00:00-05:00          2.35          2.40         2.26
2024-10-31 16:00:00-04:00          1.60          1.64         2.28
2024-08-01 16:00:00-04:00          1.34          1.40         4.30
2024-05-02 16:00:00-04:00          1.51          1.53         1.46
2024-02-01 16:00:00-05:00          2.10          2.18         3.66
2023-11-02 16:00:00-04:00          1.39          1.46         4.93
2023-08-03 16:00:00-04:00          1.19          1.

In [7]:
import sys
sys.path.insert(0, "..")

from src.data.price_loader import fetch_all_earnings_dates
from src.data.event_builder import UNIVERSE

events = fetch_all_earnings_dates(UNIVERSE)
print(events.shape)
print(events.head(10))
print("\nEvents per ticker:")
print(events.groupby("ticker").size().sort_values())

Fetching AAPL...
Fetching MSFT...
Fetching GOOGL...
Fetching AMZN...
Fetching META...
Fetching NVDA...
Fetching AMD...
Fetching INTC...
Fetching TSLA...
Fetching NFLX...
Fetching ORCL...
Fetching CRM...
Fetching ADBE...
Fetching QCOM...
Fetching TXN...

Saved 375 earnings events to D:\MathForDevs\Data Science\earnings-signals\data\processed\earnings_dates.csv
(375, 5)
  ticker earnings_date  eps_estimate  reported_eps  surprise_pct
0   AAPL    2026-04-30          1.94           NaN           NaN
1   AAPL    2026-01-29          2.67          2.84          6.25
2   AAPL    2025-10-30          1.77          1.85          4.52
3   AAPL    2025-07-31          1.43          1.57          9.48
4   AAPL    2025-05-01          1.63          1.65          1.50
5   AAPL    2025-01-30          2.35          2.40          2.26
6   AAPL    2024-10-31          1.60          1.64          2.28
7   AAPL    2024-08-01          1.34          1.40          4.30
8   AAPL    2024-05-02          1.51        

In [8]:
import pandas as pd

events["earnings_date"] = pd.to_datetime(events["earnings_date"])

# Apply the same filters as event_builder.py
mask_range = (
    (events["earnings_date"] >= "2019-01-01") &
    (events["earnings_date"] <= "2023-12-31")
)
mask_covid = (
    (events["earnings_date"] >= "2020-03-15") &
    (events["earnings_date"] <= "2020-06-30")
)

events_filtered = events[mask_range & ~mask_covid].reset_index(drop=True)

print(f"Total events:    {len(events)}")
print(f"After filtering: {len(events_filtered)}")
print(f"\nEvents per ticker:")
print(events_filtered.groupby("ticker").size().sort_values())
print(f"\nDate range: {events_filtered.earnings_date.min().date()} → {events_filtered.earnings_date.max().date()}")

Total events:    375
After filtering: 210

Events per ticker:
ticker
AAPL     14
ADBE     14
AMD      14
AMZN     14
CRM      14
GOOGL    14
INTC     14
META     14
MSFT     14
NFLX     14
NVDA     14
ORCL     14
QCOM     14
TSLA     14
TXN      14
dtype: int64

Date range: 2020-07-16 → 2023-12-13


In [9]:
import pickle
import pandas as pd
from pathlib import Path

with open("../data/raw/motley-fool-data.pkl", "rb") as f:
    raw = pickle.load(f)

UNIVERSE = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "AMD", "INTC", "TSLA", "NFLX",
    "ORCL", "CRM", "ADBE", "QCOM", "TXN",
]

subset = raw[raw["ticker"].isin(UNIVERSE)].copy()
print(f"Universe filter: {len(raw)} → {len(subset)}")

# Strip time and timezone, then parse with mixed format
cleaned_dates = subset["date"].str.replace(
    r",?\s+\d+:\d+\s+[ap]\.m\.\s+ET", "", regex=True
).str.strip()

subset["earnings_date"] = pd.to_datetime(
    cleaned_dates, format="mixed", dayfirst=False
).dt.normalize()

# Filter to analysis window
mask_range = (
    (subset["earnings_date"] >= "2019-01-01") &
    (subset["earnings_date"] <= "2023-12-31")
)
mask_covid = (
    (subset["earnings_date"] >= "2020-03-15") &
    (subset["earnings_date"] <= "2020-06-30")
)
subset = subset[mask_range & ~mask_covid].reset_index(drop=True)
print(f"Date filter: → {len(subset)} transcripts")

subset = subset[["ticker", "earnings_date", "transcript"]].copy()

out_path = Path("../data/processed/transcripts.csv")
subset.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print(f"\nTranscripts per ticker:")
print(subset.groupby("ticker").size().sort_values())
print(f"\nFile size: {out_path.stat().st_size / 1024 / 1024:.1f} MB")

Universe filter: 18755 → 326
Date filter: → 274 transcripts
Saved to ..\data\processed\transcripts.csv

Transcripts per ticker:
ticker
ADBE      3
TXN       7
META      9
MSFT     10
NFLX     10
NVDA     10
AMD      11
QCOM     11
ORCL     12
CRM      13
INTC     16
AAPL     37
AMZN     38
GOOGL    40
TSLA     47
dtype: int64

File size: 12.7 MB


In [10]:
import pandas as pd

events = pd.read_csv("../data/processed/earnings_dates.csv", parse_dates=["earnings_date"])
transcripts = pd.read_csv("../data/processed/transcripts.csv", parse_dates=["earnings_date"])

# Apply same filters to events
mask_range = (
    (events["earnings_date"] >= "2019-01-01") &
    (events["earnings_date"] <= "2023-12-31")
)
mask_covid = (
    (events["earnings_date"] >= "2020-03-15") &
    (events["earnings_date"] <= "2020-06-30")
)
events_filtered = events[mask_range & ~mask_covid].reset_index(drop=True)

# Match on ticker + date (within 3 days tolerance)
matched = 0
unmatched = []

for _, event in events_filtered.iterrows():
    ticker = event["ticker"]
    date = event["earnings_date"]
    
    candidates = transcripts[
        (transcripts["ticker"] == ticker) &
        (abs(transcripts["earnings_date"] - date) <= pd.Timedelta(days=3))
    ]
    
    if len(candidates) > 0:
        matched += 1
    else:
        unmatched.append((ticker, date.date()))

print(f"Events:    {len(events_filtered)}")
print(f"Matched:   {matched}")
print(f"Unmatched: {len(unmatched)}")
print(f"\nSample unmatched:")
for t, d in unmatched[:10]:
    print(f"  {t} {d}")

Events:    210
Matched:   123
Unmatched: 87

Sample unmatched:
  AAPL 2023-11-02
  AAPL 2023-08-03
  AAPL 2023-05-04
  MSFT 2023-10-24
  MSFT 2023-07-25
  MSFT 2023-04-25
  MSFT 2020-10-27
  MSFT 2020-07-22
  GOOGL 2023-10-24
  GOOGL 2023-07-25


In [13]:
# Build the matched dataset — the actual analysis sample
matched_rows = []

for _, event in events_filtered.iterrows():
    ticker = event["ticker"]
    date = event["earnings_date"]
    
    candidates = transcripts[
        (transcripts["ticker"] == ticker) &
        (abs(transcripts["earnings_date"] - date) <= pd.Timedelta(days=3))
    ]
    
    if len(candidates) > 0:
        transcript_text = candidates.iloc[0]["transcript"]
        matched_rows.append({
            "ticker": ticker,
            "earnings_date": date,
            "eps_estimate": event["eps_estimate"],
            "reported_eps": event["reported_eps"],
            "surprise_pct": event["surprise_pct"],
            "transcript": transcript_text,
        })

matched_df = pd.DataFrame(matched_rows)
matched_df.to_csv("../data/processed/matched_events.csv", index=False)

print(f"Matched events: {len(matched_df)}")
print(f"\nPer ticker:")
print(matched_df.groupby("ticker").size().sort_values())
print(f"\nDate range: {matched_df.earnings_date.min().date()} → {matched_df.earnings_date.max().date()}")
print(f"\nFile size: {(Path('../data/processed/matched_events.csv')).stat().st_size / 1024 / 1024:.1f} MB")

Matched events: 123

Per ticker:
ticker
ADBE      2
TXN       4
META      6
CRM       7
NVDA      7
ORCL      8
NFLX      9
AMD       9
MSFT      9
TSLA      9
QCOM     10
AMZN     10
INTC     11
GOOGL    11
AAPL     11
dtype: int64

Date range: 2020-07-22 → 2023-02-02

File size: 5.8 MB


In [14]:
import sys
sys.path.insert(0, "..")
import pandas as pd

from src.nlp.sentiment import compute_sentiment, load_lm_dictionary

matched = pd.read_csv("../data/processed/matched_events.csv")
lm_dict = load_lm_dictionary()

# Test on first transcript
sample = matched.iloc[0]
scores = compute_sentiment(sample["transcript"], lm_dict)

print(f"Ticker: {sample['ticker']} | Date: {sample['earnings_date']}")
print(f"VADER compound:  {scores['vader_compound']:.4f}")
print(f"LM score:        {scores['lm_score']:.4f}")
print(f"LM positive:     {scores['lm_positive_count']}")
print(f"LM negative:     {scores['lm_negative_count']}")
print(f"LM uncertain:    {scores['lm_uncertain_count']}")
print(f"Total words:     {scores['lm_word_count']}")

Ticker: AAPL | Date: 2023-02-02
VADER compound:  1.0000
LM score:        0.2841
LM positive:     52
LM negative:     27
LM uncertain:    8
Total words:     8374
